[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahirtonlopes/AI-and-Robotics-Trends/blob/main/Notebooks/Demo4_Attack_Detection.ipynb)

# Demo 4 - Demonstração - Aprendizagem Supervisionada

### Cybersecurity MBA - Artificial Intelligence & Robotics Trends (SEG)

### Algoritmo para classificação de cyber ataque via Árvores de Decisão.

### Prof. Dr. Ahirton Lopes (profahirton.lopes@fiap.com.br)

Ref. https://research.unsw.edu.au/projects/unsw-nb15-dataset

In [ ]:
!pip install graphviz

In [ ]:
!pip install dtreeviz

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

sns.set_context('notebook')
sns.set_style('white')

import dtreeviz

## Preparação de Dados

In [ ]:
'''
# Upload de nosso arquivo.

from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))
'''

In [ ]:
training = pd.read_csv("https://raw.githubusercontent.com/ahirtonlopes/AI-and-Robotics-Trends/main/Datasets/UNSW_NB15_training-set.csv")
testing = pd.read_csv("https://raw.githubusercontent.com/ahirtonlopes/AI-and-Robotics-Trends/main/Datasets/UNSW_NB15_testing-set.csv")
print("training ",training.shape)
print("testing ",testing.shape)

In [ ]:
all(training.columns == testing.columns)

In [ ]:
df = pd.concat([training,testing]).drop('id',axis=1)
df = df.reset_index(drop=True)

df.head()

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.attack_cat.unique()

## Tratando dados categóricos

In [ ]:
for col in ['proto', 'service', 'state']:
    df[col] = df[col].astype('category').cat.codes

df['attack_cat'] = df['attack_cat'].astype('category')

In [ ]:
df.head()

## Visualizando categorias de ataques

In [ ]:
validAttacks = df[df['label']==1]['attack_cat'].value_counts()
print(validAttacks)

plt.figure(figsize = (15,8))
plt.pie(validAttacks,labels = validAttacks.index, autopct = '%1.1f%%',explode = [0,0,0,0,0,0.2,0.2,0.2,0.2,1.2])
plt.show()

* Ataques genéricos e exploits são os mais comuns.

## Dividindo nossos dados!

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns = ['attack_cat', 'label'])
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=11)

feature_names = list(X.columns)

print("X_train shape: ", X_train.shape)
print("y_train shape: ", y_train.shape)
print("X_test shape: ", X_test.shape)
print("y_test shape: ", y_test.shape)

## Modelo de Árvore de Decisão:

* Usando um modelo de árvore de decisão para criar um conjunto de critérios para detectar ataques cibernéticos. Conduzindo uma pesquisa de grade para otimizar o modelo para maximizar o recall. A taxa de recall deve ser alta para criar a primeira camada de proteção.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [2, 4],
    'min_samples_split': [2, 4],
    'min_samples_leaf': [1, 2]
}

dt = DecisionTreeClassifier()

grid_search = GridSearchCV(dt, param_grid, cv=5, scoring='recall')
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best recall score:", grid_search.best_score_)

In [ ]:
from sklearn.metrics import recall_score
from sklearn.metrics import accuracy_score

clf=grid_search.best_estimator_
clf.fit(X_train,y_train)
y_pred = clf.predict(X_test)

recall = recall_score(y_test, y_pred)
print("Recall: ", recall)

## Visualizando as regras

In [ ]:
from sklearn.tree import export_text
import dtreeviz

print(":::::::> The RULES FOR HIGH RECALL RATE <::::::: \n" ,export_text(clf,feature_names=feature_names))

viz_model = dtreeviz.model(clf,
                           X_train=X_train, y_train=y_train,
                           feature_names=feature_names)
viz_model.view()

* A partir da árvore de decisão, visualizamos a regra da árvore como sttl<=61 e sttl>61

## Filtrando dados para ataques potenciais

In [ ]:
X_test = X_test.reset_index(drop=True)

rules= "(sttl <= 61.00 & sinpkt<= 0.00) | (sttl >  61.00 )"

ind = X_test.query(rules).index

X_test_2 = X_test.loc[ind,:]
y_test_2 = y_test[ind]

print(X_test.shape)
print(X_test_2.shape)
print("filtered data" , (1- np.round(X_test_2.shape[0] / X_test.shape[0],2))*100, "%")

* O sistema baseado em regras filtrou 23% do tráfego de rede para análise posterior, demonstrando sua eficácia na detecção de atividade de rede não ameaçadora. Agora, detectaremos ataques cibernéticos aplicando o modelo treinado aos dados de teste filtrados.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score

In [ ]:
def model_evaluation(model):
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test_2)

    accuracy = accuracy_score(y_test_2, y_pred)
    recall = recall_score(y_test_2, y_pred)
    precision = precision_score(y_test_2, y_pred)
    print("Recall: ", recall)
    print("Precision: ", precision)
    print("Accuracy: ", accuracy)

    cross = pd.crosstab(pd.Series(y_test_2, name='Actual'), pd.Series(y_pred, name='Predicted'))
    plt.figure(figsize=(5, 5))
    sns.heatmap(cross, annot=True,fmt='d', cmap="YlGnBu")
    plt.show()

    return {'Recall' : recall}

In [ ]:
results = {}

## Modelo Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=11)

results['Random Forest Model'] = model_evaluation(rf)

In [ ]:
from sklearn.tree import export_text
feature_names = list(X.columns)

print(export_text(rf.estimators_[99],feature_names=feature_names))

In [ ]:
from xgboost import XGBClassifier

xgbc = XGBClassifier()

results['XGBoost Classifier'] = model_evaluation(xgbc)

In [ ]:
viz_model = dtreeviz.model(xgbc,tree_index=1,
                           X_train=X_train, y_train=y_train,
                           feature_names=feature_names)
viz_model.view()

In [ ]:
from lightgbm import LGBMClassifier

lgbc = LGBMClassifier()

results['Light GBM Classifier'] = model_evaluation(lgbc)

In [ ]:
import lightgbm

lightgbm.plot_tree(lgbc,figsize = (20,12))
plt.show()

In [ ]:
comparision = pd.DataFrame(results)
comparision

In [ ]:
# Comparar três modelos com UM número cada não basta: qual a incerteza?
# Intervalo de confiança do recall por bootstrap sobre as predições no conjunto filtrado.
# (O teste de Wilcoxon sobre três números, usado antes, não tem validade estatística: com n=3 o menor p-valor possível é 0,25.)
import numpy as np
from sklearn.metrics import recall_score

rng = np.random.default_rng(11)
modelos = {'Random Forest': rf, 'XGBoost': xgbc, 'LightGBM': lgbc}
for nome, m in modelos.items():
    pred = m.predict(X_test_2)
    n = len(y_test_2)
    rec = []
    for _ in range(300):
        i = rng.integers(0, n, n)
        rec.append(recall_score(y_test_2[i], pred[i]))
    lo, hi = np.percentile(rec, [2.5, 97.5])
    print(f"{nome:14s} recall = {recall_score(y_test_2, pred):.4f}   IC 95% [{lo:.4f}, {hi:.4f}]")
print("\nAtenção: este intervalo mede só o erro de amostragem do conjunto de teste. A variação entre treinos (semente, partição) não está incluída; para eleger um modelo, repita com validação cruzada (ver exercício 4).")

## Lições e cuidados

- **Prevalência artificial:** no UNSW-NB15, 64% das conexões são ataques. Em uma rede real a proporção é muito menor, e a **precision cai**. Reamostre para 1% de ataques e veja.
- **Atributo de laboratório:** a regra usa `sttl` (TTL), que provavelmente reflete o ambiente de captura (hipótese; não foi testado em outra rede). Um atacante também controla esse valor.
- **Comparar com cuidado:** o bootstrap acima mede só o erro de amostragem do teste. Diferenças de cerca de 1 ponto entre ensembles só valem como 'melhor' depois de medir a variação entre treinos (validação cruzada, várias sementes).

## Para praticar

1. **Limiar e custo:** troque a árvore por `LogisticRegression` e monte a curva precision-recall (`precision_recall_curve`).
2. **Prevalência:** reamostre o teste para 1% de ataques e recalcule precision e recall.
3. **Quebre o atalho:** remova `sttl` e treine de novo. Quais atributos assumem o lugar?
4. **Comparar com cuidado:** use `cross_val_score(..., scoring='recall', cv=5)` nos três ensembles.